# Partie 4 : factoriser son réseau

Ce notebook suit la **partie 4** de l'énoncé : <https://bonben.github.io/ensta-tp-deep-learning/04_factorisation/>.

Sur Colab, passez d'abord sur GPU : *Exécution > Modifier le type d'exécution > GPU T4*.

Comme la semaine dernière : les cellules **TODO** sont à écrire, et la cellule **Vérification**
qui suit doit afficher `OK`.

## 0. Le code du réseau et les données (fourni)

In [ ]:
import os
import tarfile
import urllib.request

SITE = "https://bonben.github.io/ensta-tp-deep-learning"
CIFAR_URL = "https://github.com/bonben/ensta-tp-deep-learning/releases/download/cifar10/cifar-10-python.tar.gz"

if not os.path.exists("resnet.py"):
    urllib.request.urlretrieve(f"{SITE}/code/resnet.py", "resnet.py")
if not os.path.exists("data/cifar-10-batches-py"):
    urllib.request.urlretrieve(CIFAR_URL, "cifar.tar.gz")
    with tarfile.open("cifar.tar.gz") as tar:
        tar.extractall("data", filter="data")
    os.remove("cifar.tar.gz")
print("resnet.py et CIFAR-10 sont prêts")

## Imports, données et outils (fourni)

In [ ]:
import time
import torch
import torch.nn as nn
import torchvision
from torchvision.transforms import v2
from torch.utils.flop_counter import FlopCounterMode
import matplotlib.pyplot as plt
from resnet import ResNet, BasicBlock

torch.multiprocessing.set_start_method("fork", force=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Calcul sur :", device)

MEAN = (0.4914, 0.4822, 0.4465)
STD = (0.2470, 0.2435, 0.2616)
transform_train = v2.Compose([v2.RandomCrop(32, padding=4), v2.RandomHorizontalFlip(),
                              v2.ToImage(), v2.ToDtype(torch.float32, scale=True), v2.Normalize(MEAN, STD)])
transform_val = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True), v2.Normalize(MEAN, STD)])
train_set = torchvision.datasets.CIFAR10("./data", train=True, download=True, transform=transform_train)
val_set = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=transform_val)
train_loader = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True, num_workers=2)
val_loader = torch.utils.data.DataLoader(val_set, batch_size=256, shuffle=False, num_workers=2)


def count_params(model):
    """Nombre de paramètres appris."""
    return sum(p.numel() for p in model.parameters())


def count_macs(model):
    """Nombre de multiplications-additions (MAC) pour classer UNE image."""
    model.eval()
    with FlopCounterMode(display=False) as counter:
        model(torch.zeros(1, 3, 32, 32, device=next(model.parameters()).device))
    return counter.get_total_flops() // 2


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, 100 * correct / total


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            total_loss += criterion(outputs, labels).item() * labels.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, 100 * correct / total


def fit(model, epochs, lr=0.1):
    """Entraîne le modèle (même recette que la baseline) et renvoie l'historique."""
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    history = {"train_acc": [], "val_acc": []}
    for epoch in range(epochs):
        t0 = time.time()
        _, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        _, val_acc = evaluate(model, val_loader, criterion)
        scheduler.step()
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        print(f"époque {epoch + 1}/{epochs} | train {train_acc:.1f} % | val {val_acc:.1f} % | {time.time() - t0:.0f} s")
    print(f"Précision de validation au DERNIER epoch : {history['val_acc'][-1]:.2f} %")
    return history


baseline = ResNet(BasicBlock, [2, 2, 2, 2], width=64).to(device)
print(f"ResNet18 : {count_params(baseline):,} paramètres, {count_macs(baseline):,} MAC par image")

## 1. Compter à la main

Une convolution 3×3 « pleine » de $C_{in}$ input FM vers $C_{out}$ output FM, sans biais, a
$9 \cdot C_{in} \cdot C_{out}$ paramètres.

Une convolution **séparable en profondeur** la remplace par deux étapes :

- une convolution 3×3 **par FM** (*depthwise*) : un seul filtre 3×3 pour chacune des $C_{in}$ FM ;
- une convolution 1×1 (*pointwise*) qui mélange les $C_{in}$ FM pour produire les $C_{out}$ output FM.

In [ ]:
# TODO 1 : avec C_in = C_out = 64, calculez À LA MAIN (avec des nombres, pas des formules Python)
p_pleine = ...       # paramètres de la convolution 3×3 pleine
p_depthwise = ...    # paramètres de la convolution 3×3 par FM
p_pointwise = ...    # paramètres de la convolution 1×1

In [ ]:
# Vérification 1
pleine = nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False)
assert p_pleine == count_params(pleine), "convolution pleine : recomptez"
assert p_depthwise + p_pointwise == 9 * 64 + 64 * 64, "séparable : recomptez"
print(f"séparable : {p_depthwise + p_pointwise} paramètres, {p_pleine / (p_depthwise + p_pointwise):.1f} fois moins")
print("OK")

## 2. Une convolution séparable

Le paramètre `groups` de `nn.Conv2d` découpe les FM en groupes indépendants. Avec
`groups=C_in`, chaque filtre ne voit **qu'une seule** FM : c'est la convolution par FM.

Prototype : `nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0, groups=1, bias=True)`

In [ ]:
class SeparableConv2d(nn.Module):
    """Remplace nn.Conv2d(in_ch, out_ch, 3, stride, padding=1) par une convolution séparable."""

    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        # TODO 2.a : la convolution 3×3 par FM : in_ch -> in_ch, noyau 3, stride=stride, padding=1,
        #            groups=in_ch, bias=False
        self.depthwise = ...
        # TODO 2.b : la convolution 1×1 : in_ch -> out_ch, noyau 1, bias=False
        self.pointwise = ...

    def forward(self, x):
        # TODO 2.c : appliquer depthwise puis pointwise
        return ...

In [ ]:
# Vérification 2
x = torch.randn(8, 64, 32, 32)
sep = SeparableConv2d(64, 128)
assert sep(x).shape == (8, 128, 32, 32), "forme de sortie incorrecte"
assert SeparableConv2d(64, 128, stride=2)(x).shape == (8, 128, 16, 16), "le stride ne divise pas H et W par 2"
assert count_params(sep) == 9 * 64 + 64 * 128, "nombre de paramètres incorrect (groups ? bias ?)"
print("OK")

## 3. Un bloc résiduel factorisé

Ouvrez `resnet.py` (panneau *Fichiers*) : un `BasicBlock` contient deux convolutions 3×3,
`self.conv1` et `self.conv2`. On crée un nouveau bloc qui **hérite** de `BasicBlock` (il en
garde tout : BatchNorm, raccourci, `forward`) et on **remplace seulement ces deux convolutions**.

In [ ]:
class SepBasicBlock(BasicBlock):
    def __init__(self, in_planes, planes, stride=1):
        super().__init__(in_planes, planes, stride)   # construit le BasicBlock normal
        # TODO 3 : remplacer self.conv1 et self.conv2 par des SeparableConv2d.
        #   Dans resnet.py : conv1 va de in_planes à planes avec le stride du bloc,
        #                    conv2 va de planes à planes avec un stride de 1.


model_sep = ResNet(SepBasicBlock, [2, 2, 2, 2], width=64).to(device)

In [ ]:
# Vérification 3
p_base, p_sep = count_params(baseline), count_params(model_sep)
m_base, m_sep = count_macs(baseline), count_macs(model_sep)
print(f"{'':20s}{'paramètres':>14s}{'MAC par image':>16s}")
print(f"{'ResNet18':20s}{p_base:>14,}{m_base:>16,}")
print(f"{'ResNet18 factorisé':20s}{p_sep:>14,}{m_sep:>16,}")
print(f"{'rapport':20s}{p_base / p_sep:>14.1f}{m_base / m_sep:>16.1f}")
assert p_sep == 1_439_626, "avez-vous bien remplacé les DEUX convolutions de chaque bloc ?"
print("OK")

**Q1.** Le rapport est d'environ 7,8, alors qu'une convolution 64 → 64 est divisée par 7,9 et
une 512 → 512 par 8,8. Où sont les paramètres qui n'ont pas été factorisés ?
(Regardez `resnet.py` : la première convolution, le raccourci, la couche finale.)

## 4. Entraîner le réseau factorisé

Lancez l'entraînement sur **15 époques**. Notez la précision **au dernier epoch** et la durée
d'une époque.

In [ ]:
history_sep = fit(model_sep, epochs=15)

**Q2.** Référence, mesurée avec la même recette : le ResNet18 fait **90,6 %** en 15 époques.
Combien de points avez-vous perdus pour un réseau près de 8 fois plus petit ?

**Q3.** Une époque est-elle 8 fois plus rapide ? Pourquoi, d'après vous ?
(Pensez au cours d'architecture : que fait le GPU pendant une convolution par FM ?)

## 5. Le fil rouge : votre plus petit réseau

Objectif du module : **le moins de paramètres possible, avec au moins 90 % en validation**.

Vos trois leviers :

- **la largeur** : `width=64` (défaut), 32, 16… Les canaux de chaque étage valent `width`,
  `2·width`, `4·width`, `8·width` ;
- **la profondeur** : `[2, 2, 2, 2]` = nombre de blocs dans chacun des 4 étages ;
- **la factorisation** : `SepBasicBlock` ou `BasicBlock`.

Pour chaque essai : affichez paramètres et MAC **avant** d'entraîner, entraînez, et reportez le
résultat dans le **tableau partagé de la promo**.

In [ ]:
# Votre essai : changez le bloc, la profondeur et la largeur
model = ResNet(SepBasicBlock, [2, 2, 2, 2], width=32).to(device)
print(f"{count_params(model):,} paramètres, {count_macs(model):,} MAC par image")

In [ ]:
history = fit(model, epochs=15)